In [ ]:
import torch
import torch.nn as nn
from functools import partial
import torch.utils.checkpoint as checkpoint
import torch.nn.functional as F
import os
import scipy.io as io
import sys
import argparse

import src.utils.utils as utils

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(dev)
print(f"Using device: {dev}")

dataset = "urban"
data = io.loadmat(f"datasets/{dataset}.mat")
Y_flat = torch.tensor(data["Y"], dtype=torch.float32)
Y_flat = utils.normalise(Y_flat, dim=0)
E = torch.tensor(data["E"])
B, c, N = E.shape[0], E.shape[1], Y_flat.shape[1]

Y = utils.oneD_to_2d(Y_flat)
H = Y.shape[1]
Y = Y[:, :64, :64].unsqueeze(0)

/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may hav

In [ ]:
sys.path.append("/home/ids/edabier/HSU/HyperSIGMA")
from HyperspectralUnmixing.models.model import SpecViT, SpatViT
from HyperspectralUnmixing.func import SumToOne, weightConstraint
import numpy as np

class Decoder(nn.Module):
    def __init__(self, num_em, channels, kernel=1):
        super(Decoder, self).__init__()
        padding = kernel //2
        self.decoder =nn.Conv2d(in_channels=num_em,
                                out_channels=channels,
                                kernel_size=kernel,stride=1,padding=padding, bias=False)
        self.relu = nn.ReLU()

    def forward(self, code):
        code = self.relu(self.decoder(code))
        return code

    def getEndmembers(self):
        constraints = weightConstraint()
        self.decoder.apply(constraints)
        return self.decoder.weight.data

class HyperSIGMA_Unmix(torch.nn.Module):
    def __init__(self, patch_size, channels, seg_patches, NUM_TOKENS, embed_dim, num_em, scale):
        super(HyperSIGMA_Unmix, self).__init__()
        self.patch_size = patch_size
        self.spat_encoder = SpatViT(img_size=patch_size,
            in_chans=channels,
            use_checkpoint=True,
            patch_size=seg_patches,
            drop_path_rate=0.1, out_indices=[3, 5, 7, 11], embed_dim=768,
            depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True, qk_scale=None,
            drop_rate=0., attn_drop_rate=0., use_abs_pos_emb=False, n_points=8
        )
        self.spec_encoder = SpecViT(
            NUM_TOKENS=NUM_TOKENS,
            img_size=patch_size,
            in_chans=channels,
            drop_path_rate=0.1,
            out_indices=[3, 5, 7, 11],
            embed_dim=768,
            depth=12,
            num_heads=12,
            mlp_ratio=4,
            qkv_bias=True,
            qk_scale=None,
            drop_rate=0.,
            attn_drop_rate=0.,
            use_checkpoint=False,
            use_abs_pos_emb=False,
            interval=3
        )

        self.conv_features1 = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)
        self.fc_spec1 = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )
        self.conv_features2 = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)
        self.fc_spec2 = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )
        self.conv_features3 = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)
        self.fc_spec3 = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )
        self.conv_features4 = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)
        self.fc_spec4 = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.conv1 =nn.Sequential(
            nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=1),
            nn.LeakyReLU(0.02),# nn.ReLU(),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=3, padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=3, padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )

        self.conv2_ = nn.Sequential(
            nn.Conv2d(NUM_TOKENS*2, NUM_TOKENS, kernel_size=3, padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.conv3_ = nn.Sequential(
            nn.Conv2d(NUM_TOKENS*2, NUM_TOKENS, kernel_size=3, padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.conv4_ = nn.Sequential(
            nn.Conv2d(NUM_TOKENS*2, NUM_TOKENS, kernel_size=3, padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS),
            nn.Dropout(0.2),
        )
        self.smooth = nn.Sequential(
            nn.Conv2d(NUM_TOKENS*4, NUM_TOKENS*2, kernel_size=(3, 3), padding=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(NUM_TOKENS*2),
            nn.Dropout(0.2),
            nn.Conv2d(NUM_TOKENS*2, NUM_TOKENS, kernel_size=(1, 1)) 
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(NUM_TOKENS, num_em, kernel_size=1),
            nn.LeakyReLU(0.02),
            nn.BatchNorm2d(num_em),
            nn.Dropout(0.2),
        )

        self.sumtoone = SumToOne(scale)
        self.decoder = Decoder(num_em=num_em, channels=channels)

    def _upsample_add(self, x, y):
        '''Upsample and add two feature maps.
        Args:
        x: (Variable) top feature map to be upsampled.
        y: (Variable) lateral feature map.
        Returns:
        (Variable) added feature map.
        Note in PyTorch, when input size is odd, the upsampled feature map
        with `F.upsample(..., scale_factor=2, mode='nearest')`
        maybe not equal to the lateral feature map size.
        e.g.
        original input size: [N,_,15,15] ->
        conv2d feature map size: [N,_,8,8] ->
        upsampled feature map size: [N,_,16,16]
        So we choose bilinear upsample which supports arbitrary output sizes.
        '''
        _, _, H, W = y.size()
        return F.interpolate(x, size=(H, W), mode='bilinear', align_corners=True) + y

    @staticmethod
    def weights_init(m):
        if type(m) == nn.Conv2d:
            nn.init.kaiming_normal_(m.weight.data)
            nn.init.normal_(m.weight.data, mean=0.0, std=0.3)

    def forward_fusion(self, x):
        # x: (b, c, h, w)
        # ts:(b, c)
        # spat
        b, _, h, w = x.shape
        img_features = self.spat_encoder(x)

        img_fea = []
        ops = [self.conv_features1, self.conv_features2, self.conv_features3, self.conv_features4]
        for i in range(len(ops)):
            img_fea.append(ops[i](img_features[i+1]))

        spec_features = self.spec_encoder(x)
        spec_feature = spec_features[-1]
        spec_feature = self.pool(spec_feature).view(b, -1) # b, c

        spec_weights = []
        ops_ = [self.fc_spec1, self.fc_spec2, self.fc_spec3, self.fc_spec4]
        for i in range(len(ops_)):
            spec_weights.append((ops_[i](spec_feature)).view(b, -1, 1, 1))
        ss_feature = []
        ss_feature.append(x)
        for i in range(4):
            ss_feature.append((1 + spec_weights[i]) * img_fea[i])
        return ss_feature

    def getAbundances(self, x):
        H, W = x.shape[2], x.shape[3]
        x = self.forward_fusion(x) # x: list : 5
        p4 = self.conv1(x[4])
        p3 = self.conv2(x[3])
        p2 = self.conv3(x[2])
        p1 = self.conv4(x[1])
        p1 = torch.cat([p1,p2,p3,p4], dim=1)

        p1 = F.interpolate(p1, size=(H, W), mode='bilinear', align_corners=True)
        p1 = self.smooth(p1)
        x = self.conv5(p1)
        abunds = self.sumtoone(x)
        abunds = abunds
        return abunds

    def forward(self, patch):
        abunds = self.getAbundances(patch)
        output = self.decoder(abunds)
        return abunds,output

    def getEndmembers(self):
        endmembers = self.decoder.getEndmembers()
        if endmembers.shape[2] > 1:
            endmembers = np.squeeze(endmembers).mean(axis=2).mean(axis=2)
        else:
            endmembers = np.squeeze(endmembers)
        return endmembers

img_size, embed_dim, patch_size, NUM_TOKENS, scale = 64, 768, 2, 64, 1
hypersigma = HyperSIGMA_Unmix(patch_size=img_size, channels=B, seg_patches=patch_size, NUM_TOKENS=NUM_TOKENS, embed_dim=embed_dim, num_em=c, scale=scale)

spat_path = "/home/ids/edabier/HSU/HyperSIGMA/HyperspectralUnmixing/data/spat-vit-base-ultra-checkpoint-1599.pth"
spec_path = "/home/ids/edabier/HSU/HyperSIGMA/HyperspectralUnmixing/data/spec-vit-base-ultra-checkpoint-1599.pth"
Spat_pernet = torch.load(spat_path, map_location=torch.device('cpu'), weights_only=False)
Spat_pernet = Spat_pernet['model']
for k in list(Spat_pernet.keys()):
    if 'patch_embed.proj' in k:
        del Spat_pernet[k]
for k in list(Spat_pernet.keys()):
    k_ = 'spat_encoder.' + k
    Spat_pernet[k_] = Spat_pernet.pop(k)

Spec_pernet = torch.load(spec_path, map_location=torch.device('cpu'), weights_only=False)
Spec_pernet = Spec_pernet['model']
for k in list(Spec_pernet.keys()):
    if 'spec' in k:
        del Spec_pernet[k]
    if 'spat' in k:
        del Spec_pernet[k]
for k in list(Spec_pernet.keys()):
    k_ = 'spec_encoder.' + k
    Spec_pernet[k_] = Spec_pernet.pop(k)

model_params = hypersigma.state_dict()
same_parsms = {k: v for k, v in Spat_pernet.items() if k in model_params.keys()}
model_params.update(same_parsms)
hypersigma.load_state_dict(model_params)

same_parsms = {k: v for k, v in Spec_pernet.items() if k in model_params.keys()}
model_params.update(same_parsms)
hypersigma.load_state_dict(model_params)
print(Y.shape)
features = hypersigma.forward_fusion(Y)
for f in features:
    print(f.shape)

In [2]:
sys.path.append("/home/ids/edabier/HSU/HyperSIGMA")
from HyperspectralUnmixing.models.model import HyperSIGMA_Unmix

img_size, embed_dim, patch_size, num_heads, depth, mlp_ratio, qkv_bias, qk_scale, drop_rate, attn_drop_rate = 64, 768, 2, 12, 12, 4., True, None, 0., 0.
drop_path_rate, init_values, interval, restart_regression, n_points, out_indices, NUM_TOKENS = 0.1, None, 3, True, 8, [3,5,7,11], 64
norm_layer = partial(nn.LayerNorm, eps=1e-6)
dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]

parser = argparse.ArgumentParser('argument for training')
parser.add_argument('--seed', type=int, default=1)
parser.add_argument('--batch_size', default=64, type=int, metavar='C', help='C')
model_name = 'HyperSIGMA_Unmix'
parser.add_argument('--model_name', type=str, default=model_name, help='')
parser.add_argument('--img_size', type=int, default=img_size, help='')
parser.add_argument('--patch_size', type=int, default=img_size, help='')
parser.add_argument('--patches', type=int, default=img_size, help='')
parser.add_argument('--seg_patches', type=int, default=patch_size, help='')
parser.add_argument('--use_checkpoint', type=bool, default=True, help='')
path = os.getcwd()
path = path + '/data/'
pretrain_Spatpath = ''
pretrain_Specpath = ''
pretrain_Spatpath = path + 'spat-vit-base-ultra-checkpoint-1599.pth'
pretrain_Specpath = path + 'spec-vit-base-ultra-checkpoint-1599.pth'
parser.add_argument('--pretrain_Spatpath', type=str,default= pretrain_Spatpath)
parser.add_argument('--pretrain_Specpath', type=str,default= pretrain_Specpath)
parser.add_argument('--interval', type=int, default=3, help='')
parser.add_argument('--embed_dim', type=int, default=768, help='embed_dim')
parser.add_argument('--NUM_TOKENS', type=int, default=NUM_TOKENS, help='NUM_TOKENS')
parser.add_argument('--kernel', default=1, type=int, metavar='C', help='C')
parser.add_argument('--scale', default=1, type=float, metavar='C', help='C')
parser.add_argument('--num_patches', default=400, type=int, metavar='C', help='C')
parser.add_argument('--channels', default=B, type=int, metavar='C', help='C')
parser.add_argument('--H', default=H, type=int, metavar='H', help='H')
parser.add_argument('--W', default=H, type=int, metavar='W', help='W')
parser.add_argument('--num_em', default=c, type=int, metavar='C', help='C')
args, unknown = parser.parse_known_args()

model = HyperSIGMA_Unmix(args)

/home/ids/edabier/.local/lib/python3.10/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 28] No space left on device.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))


/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
import random
import numpy as np
from torch.utils import data as Data
from HyperspectralUnmixing.func import mirror_hsi, get_train_data

def get_train_patch_seed_i(img_size, img, train_number, batch_size,seed_i, n_workers=1):
    seed=seed_i
    np.random.seed(seed)
    random.seed(seed)

    height, width, band=img.shape
    data_label = np.ones([height, width])
    position = np.array(np.where(data_label == 1)).transpose(1, 0)
    selected_i = np.random.choice(position.shape[0], int(train_number), replace=False)
    selected_i =position[selected_i]

    TR = np.zeros(data_label.shape)
    for i in range(int(train_number)):
        TR[selected_i[i][0], selected_i[i][1]] = 1
    total_pos_train = np.argwhere(TR == 1)
    mirror_img = mirror_hsi(height, width, band, img, patch=img_size)
    img_train= get_train_data(mirror_img, band, total_pos_train,
                                                         patch=img_size, band_patch=1)
    # load data
    g = torch.Generator()
    if torch.cuda.is_available():
        g = torch.Generator(device='cuda')
    img_train = torch.from_numpy(img_train.transpose(0,3,1,2)).type(torch.FloatTensor)  # [1000, 155, 5, 5]
    Label_train = Data.TensorDataset(img_train)
    label_train_loader = Data.DataLoader(Label_train, batch_size=batch_size, shuffle=True, generator=g, 
                                         num_workers=n_workers, persistent_workers=True, 
                                         pin_memory=True, prefetch_factor=2)

    return label_train_loader

label_train_loader = get_train_patch_seed_i(args.img_size, Y[0].cpu(), args.num_patches, args.batch_size, args.seed, n_workers=1)

**************************************************
patch is : 307
mirror_image shape : [468,613,307]
**************************************************


: 

In [20]:
x = torch.rand(1, B, img_size, img_size)
spat_features = model.spat_encoder(Y.unsqueeze(0))[1:]
spec_features = model.spec_encoder(Y.unsqueeze(0))
print(spat_features[0].shape, spec_features[0].shape)

/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


: 

In [ ]:
from HyperspectralUnmixing.models.SpatVit import Block

def get_reference_points(spatial_shapes, device):
    # for lvl, (H_, W_) in enumerate(spatial_shapes):
    H_, W_ = spatial_shapes[0], spatial_shapes[1]
    ref_y, ref_x = torch.meshgrid(
        torch.linspace(0.5, H_ - 0.5, H_, dtype=torch.float32, device=device),
        torch.linspace(0.5, W_ - 0.5, W_, dtype=torch.float32, device=device))
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    ref = torch.stack((ref_x, ref_y), -1)  # 每个点归一化后的坐标，尺寸(1，H_*W_，2)
    return ref

def deform_PatchInputs_func(x, seg_patch_size):
    B, c, h, w = x.shape
    b = B // 3
    spatial_shapes = torch.as_tensor([h // seg_patch_size, w // seg_patch_size],
                                     dtype=torch.long, device=x.device)
    reference_points = get_reference_points([h // seg_patch_size, w // seg_patch_size],
                                            x.device)  # [h // 8, w // 8], x.device)
    deform_inputs = [reference_points, spatial_shapes]
    return deform_inputs

class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=64, in_chans=3, embed_dim=768):
        super().__init__()

        img_size = (img_size, img_size)
        patch_size = (patch_size, patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x, **kwargs):
        B, C, H, W = x.shape
        x = self.proj(x)
        Hp, Wp = x.shape[2], x.shape[3]

        x = x.flatten(2).transpose(1, 2)
        return x, (Hp, Wp)

pos_drop = nn.Dropout(p=drop_rate)
conv_features = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)

conv1 = nn.Sequential(
    nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(NUM_TOKENS),
    nn.Dropout(0.2),
    )
conv2 = nn.Sequential(
    nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=3, padding=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(NUM_TOKENS),
    nn.Dropout(0.2),
)
smooth = nn.Conv2d(NUM_TOKENS*4, NUM_TOKENS, kernel_size=3, stride=1, padding=1)
conv3 = nn.Sequential(
    nn.Conv2d(NUM_TOKENS, c, kernel_size=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(c),
    nn.Dropout(0.2),
)

blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values, sample=((i + 1) % interval != 0),
                restart_regression=restart_regression, n_points=n_points)
            for i in range(depth)])

patch_embed = PatchEmbed(img_size=img_size, patch_size=2, in_chans=B)

x = torch.rand(1, B, img_size, img_size)
img = [x]
deform_inputs = deform_PatchInputs_func(x, 2)
x, (Hp, Wp) = patch_embed(x)
x = pos_drop(x)

features = []
for i, blk in enumerate(blocks):
    x = checkpoint.checkpoint(blk, x, Hp, Wp, deform_inputs)

    if i in out_indices:
        features.append(x)
features = list(map(lambda x: x.permute(0, 2, 1).reshape(1, -1, Hp, Wp), features))

ops = [nn.Identity(), nn.Identity(), nn.Identity(), nn.Identity()]
for i in range(len(ops)):
    features[i] = ops[i](features[i])

Feat_spat = img + features
x = []
x.append(Feat_spat[0])
ops = [conv_features, conv_features, conv_features, conv_features]
for i in range(len(ops)):
    x.append(ops[i](Feat_spat[i+1]))
img_fea = x[1:]

p4 = conv1(x[4])
p3 = conv1(x[3])
p2 = conv2(x[2])
p1 = conv2(x[1])
p1 = torch.cat([p1,p2,p3,p4], dim=1)

p1 = F.interpolate(p1, size=(64, 64), mode='bilinear', align_corners=True)
p1 = smooth(p1)
x = conv3(p1)
x.shape

torch.Size([1, 64, 32, 32])


In [3]:
from HyperSIGMA.HyperspectralUnmixing.models.SpecVit import SpecViT
import math

spec_vit = SpecViT(NUM_TOKENS=NUM_TOKENS, img_size=patch_size, in_chans=B,drop_path_rate=0.1, out_indices=[3, 5, 7, 11],
            embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True, qk_scale=None,
            drop_rate=0., attn_drop_rate=0., use_checkpoint=False, use_abs_pos_emb=False, interval=3)

def get_reference_points(spatial_shapes, device):
    H_, W_ = spatial_shapes[0], spatial_shapes[1]
    ref_y, ref_x = torch.meshgrid(
        torch.linspace(0.5, H_ - 0.5, H_, dtype=torch.float32, device=device),
        torch.linspace(0.5, W_ - 0.5, W_, dtype=torch.float32, device=device))
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    ref = torch.stack((ref_x, ref_y), -1)  # 每个点归一化后的坐标，尺寸(1，H_*W_，2)
    return ref

def deform_inputs_func(x, num_tokens):
    spatial_shapes = torch.as_tensor([int(math.sqrt(num_tokens)), int(math.sqrt(num_tokens))],
                                     dtype=torch.long, device=x.device)  # 3*2的tensor
    reference_points = get_reference_points([int(math.sqrt(num_tokens)), int(math.sqrt(num_tokens))], x.device)
    deform_inputs = [reference_points, spatial_shapes]

    return deform_inputs

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224):
        super().__init__()

        img_size = (img_size,img_size)
        num_patches = (img_size[1]) * (img_size[0])
        patch_shape = (img_size[0], img_size[1])

    def forward(self, x, **kwargs):
        x = x.flatten(2).transpose(1, 2)
        return x

norm_layer = partial(nn.LayerNorm, eps=1e-6)
patch_embed = PatchEmbed(img_size=img_size)

spec_embed = nn.AdaptiveAvgPool1d(NUM_TOKENS)
spat_map = nn.Linear(int(img_size * img_size), embed_dim)

pos_drop = nn.Dropout(p=drop_rate)

dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]  # stochastic depth decay rule

blocks = nn.ModuleList([
    Block(
        dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
        drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
        init_values=init_values, sample=((i + 1) % interval != 0),
        restart_regression=restart_regression, n_points=n_points)
    for i in range(depth)])

norm = norm_layer(embed_dim)
l = nn.Linear(embed_dim, 128, bias=False)

x = torch.rand(1, B, img_size, img_size)

deform_inputs = deform_inputs_func(x, NUM_TOKENS)
x = patch_embed(x)
print(x.shape)
x = spec_embed(x)
print(x.shape)
_, _, num_tokens = x.shape
x = x.transpose(1, 2)
x_in = x.reshape(1, num_tokens, img_size, img_size)
x = spat_map(x)
print(x.shape)
batch_size, _, embed_dim = x.size()
x = pos_drop(x)

features = []
for i, blk in enumerate(blocks):
    x = blk(x, img_size, img_size, deform_inputs)

    if i in out_indices:
        features.append(x)  # b, channels, embed_dim

ops = [l, l, l, l]
for i in range(len(ops)):
    features[i] = ops[i](features[i])

Feat_spec = features

torch.Size([1, 4096, 162])
torch.Size([1, 4096, 64])
torch.Size([1, 64, 768])


In [4]:
pool = nn.AdaptiveAvgPool1d(1)
fc_spec = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )

spec_feature = Feat_spec[-1]
spec_feature = pool(spec_feature).view(1, -1)

spec_weights = []
ops_ = [fc_spec, fc_spec, fc_spec, fc_spec]
for i in range(len(ops_)):
    spec_weights.append((ops_[i](spec_feature)).view(1, -1, 1, 1))
print(spec_weights[0].shape)

ss_feature = []
ss_feature.append(x)
for i in range(4):
    ss_feature.append((1 + spec_weights[i]) * img_fea[i])

x = ss_feature
p4 = conv1(x[4])
p3 = conv1(x[3])
p2 = conv2(x[2])
p1 = conv2(x[1])
p1 = torch.cat([p1,p2,p3,p4], dim=1)

p1 = F.interpolate(p1, size=(64, 64), mode='bilinear', align_corners=True)
p1 = smooth(p1)
x = conv3(p1)
x.shape

torch.Size([1, 64, 1, 1])


torch.Size([1, 6, 64, 64])